# Option D: Raw-Space MSE Auto-Gate

Two-stage gating for the consensus ensemble:

1. **Stage 1 (score threshold):** Same per-method score thresholds as the
   base gated approach. A gene only qualifies if its metric score passes
   the method's quality bar.
2. **Stage 2 (fit gate):** For each method x cluster, load the qualifying
   genes' *raw LT expression* and compute MSE against the constraint
   pattern. If the **median raw-space MSE** exceeds `FIT_THRESHOLD`, that
   method is silenced for that cluster.

This automates what the eye does when looking at the per-method overlay
plots: "do these genes actually sit on the pattern line?"

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import os, pickle

%matplotlib inline

# ── Load data ──────────────────────────────────────────────────────────
results = pickle.load(open('metrics_results.pkl', 'rb'))
constraint_patternsLT = results['constraint_patternsLT']
all_methodsLT         = results['all_methodsLT']
method_names          = list(all_methodsLT.keys())

cluster_names = list(constraint_patternsLT.keys())
print('Methods:', method_names)
print('Clusters:', cluster_names)

## 1. Thresholds

**Stage 1** — per-method score thresholds (same as the base gated approach).  
**Stage 2** — `FIT_THRESHOLD`: median raw-space MSE cutoff for the fit gate.

In [ ]:
# ── Stage 1: per-method score thresholds ──────────────────────────────
THRESHOLDS = {
    'Pearson':  ('>', 0.99),
    'DTW':      ('<', 0.08),
    'Cosine':   ('>', 0.999),
    'Frechet':  ('<', 0.03),
    'MSE':      ('<', 0.001),
    'sMAPE':    ('<', 0.10),
}

# ── Stage 2: raw-space fit gate ──────────────────────────────────────
FIT_THRESHOLD = 0.01   # median raw-space MSE must be below this

TOP_N = 20

print('Stage 1 thresholds:')
for m, (op, val) in THRESHOLDS.items():
    print(f'  {m:>8}: score {op} {val}')
print(f'\nStage 2 fit gate:  median raw-space MSE < {FIT_THRESHOLD}')
print(f'Top-N for ensemble: {TOP_N}')

## 2. Stage 1 — `get_qualifying_genes`

For each method, return genes whose raw metric score passes the threshold.

In [ ]:
def get_qualifying_genes(methods_dict, cluster_name, pattern_name, thresholds):
    """For each method, return the set of genes that pass its threshold.

    Returns dict: method_name -> Series of scores (only qualifying genes).
    """
    qualifying = {}
    for mname, (scores, ascending) in methods_dict.items():
        if mname not in thresholds:
            continue
        op, val = thresholds[mname]
        s = scores[cluster_name][pattern_name]
        if op == '>':
            mask = s > val
        elif op == '<':
            mask = s < val
        elif op == '>=':
            mask = s >= val
        elif op == '<=':
            mask = s <= val
        qualifying[mname] = s[mask].sort_values(ascending=ascending)
    return qualifying


# Show how many genes qualify per method x cluster
print(f'{"=" * 60}')
print(f'STAGE 1 — qualifying gene counts (LT data)')
print(f'{"=" * 60}')
for cluster_name in cluster_names:
    q = get_qualifying_genes(all_methodsLT, cluster_name, 'constraint', THRESHOLDS)
    print(f'\n  {cluster_name}:')
    for mname in method_names:
        n = len(q.get(mname, []))
        op, val = THRESHOLDS[mname]
        tag = '  ** SILENCED (stage 1) **' if n == 0 else ''
        print(f'    {mname:>8} ({op}{val}): {n:>6,} genes{tag}')

## 3. Stage 2 — `compute_fit_quality`

For each method x cluster that has any Stage 1 qualifying genes, load
those genes' raw LT expression vectors and compute MSE against the
constraint pattern. Report the **median MSE** across qualifying genes.

If median MSE > `FIT_THRESHOLD`, the method is silenced for that cluster.

In [ ]:
def compute_fit_quality(methods_dict, pat_dict, thresholds, lt=True):
    """For each method x cluster, compute median MSE between qualifying
    genes' raw LT expression and the constraint pattern."""
    x = [0, 3, 6, 9]
    suffix = 'LT' if lt else ''
    fit = {}
    for cluster_name, pattern_dict in pat_dict.items():
        gene_file = f'analysis data/gene_counts{suffix}/{cluster_name}_annotated.csv'
        gene_df = pd.read_csv(gene_file, index_col=0)
        gene_df.columns = x
        pattern_vals = np.array(pattern_dict['constraint'])

        q = get_qualifying_genes(methods_dict, cluster_name, 'constraint', thresholds)
        fit[cluster_name] = {}
        for mname, qual_scores in q.items():
            if len(qual_scores) == 0:
                fit[cluster_name][mname] = float('inf')
                continue
            # Get qualifying genes that exist in the gene df
            genes_in_df = [g for g in qual_scores.index if g in gene_df.index]
            if len(genes_in_df) == 0:
                fit[cluster_name][mname] = float('inf')
                continue
            # Compute MSE between each gene's expression and the pattern
            gene_vals = gene_df.loc[genes_in_df].values  # shape (n_genes, 4)
            mses = ((gene_vals - pattern_vals) ** 2).mean(axis=1)
            fit[cluster_name][mname] = float(np.median(mses))
    return fit


# Compute fit quality
fit_quality = compute_fit_quality(all_methodsLT, constraint_patternsLT, THRESHOLDS, lt=True)

# Print the fit quality table
print(f'{"=" * 80}')
print(f'STAGE 2 — fit quality (median raw-space MSE of qualifying genes)')
print(f'FIT_THRESHOLD = {FIT_THRESHOLD}')
print(f'{"=" * 80}')
print(f'\n{"Method":>10}  ', end='')
for cn in cluster_names:
    print(f'{cn:>22}', end='')
print()
print(f'  {"-" * (10 + 22 * len(cluster_names))}')

for mname in method_names:
    print(f'{mname:>10}  ', end='')
    for cn in cluster_names:
        val = fit_quality.get(cn, {}).get(mname, float('inf'))
        if val == float('inf'):
            cell = 'N/A (no genes)'
        else:
            passed = val < FIT_THRESHOLD
            marker = 'PASS' if passed else 'FAIL'
            cell = f'{val:.6f} [{marker}]'
        print(f'{cell:>22}', end='')
    print()

## 4. `get_active_methods` — combine both gates

A method is active for a cluster only if:
1. It has >= 1 qualifying gene (Stage 1), **and**
2. Its median raw-space MSE < `FIT_THRESHOLD` (Stage 2).

In [ ]:
def get_active_methods(methods_dict, pat_dict, thresholds, fit_quality,
                       fit_threshold, cluster_name):
    """Return (active, silenced_stage1, silenced_stage2) method lists.

    active          — methods passing both gates
    silenced_stage1 — methods with 0 qualifying genes
    silenced_stage2 — methods passing stage 1 but failing the fit gate
    """
    q = get_qualifying_genes(methods_dict, cluster_name, 'constraint', thresholds)

    active = []
    silenced_stage1 = []
    silenced_stage2 = []

    for mname in method_names:
        n_qual = len(q.get(mname, []))
        if n_qual == 0:
            silenced_stage1.append(mname)
            continue
        fq = fit_quality.get(cluster_name, {}).get(mname, float('inf'))
        if fq >= fit_threshold:
            silenced_stage2.append(mname)
            continue
        active.append(mname)

    return active, silenced_stage1, silenced_stage2


# Print summary of double-gated active methods
print(f'{"=" * 80}')
print('DOUBLE-GATED ACTIVE METHODS')
print(f'{"=" * 80}')
for cluster_name in cluster_names:
    active, sil1, sil2 = get_active_methods(
        all_methodsLT, constraint_patternsLT, THRESHOLDS,
        fit_quality, FIT_THRESHOLD, cluster_name
    )
    active_str = ", ".join(active) if active else "-- NONE --"
    print(f'\n  {cluster_name}:')
    print(f'    Active ({len(active)}):           {active_str}')
    if sil1:
        sil1_str = ", ".join(sil1)
        print(f'    Silenced stage 1 ({len(sil1)}):  {sil1_str}')
    if sil2:
        sil2_str = ", ".join(sil2)
        print(f'    Silenced stage 2 ({len(sil2)}):  {sil2_str}')

## 5. Plot qualifying genes for ACTIVE methods only

Methods silenced by either gate are excluded from the visualization.

In [ ]:
def plot_qualifying_genes_optionD(methods_dict, pat_dict, thresholds,
                                  fit_quality, fit_threshold,
                                  lt=True, max_plot=50):
    """Plot qualifying genes overlaid on pattern for double-gated active methods."""
    x = [0, 3, 6, 9]
    suffix = 'LT' if lt else ''

    for cluster_name, pattern_dict in pat_dict.items():
        gene_file = f'analysis data/gene_counts{suffix}/{cluster_name}_annotated.csv'
        gene_df = pd.read_csv(gene_file, index_col=0)
        gene_df.columns = x

        for pattern_name, pattern_vals in pattern_dict.items():
            active, sil1, sil2 = get_active_methods(
                methods_dict, pat_dict, thresholds,
                fit_quality, fit_threshold, cluster_name
            )
            if len(active) == 0:
                print(f'{cluster_name}: ALL METHODS SILENCED (stage 1: '
                      f'{", ".join(sil1)}; stage 2: {", ".join(sil2)})')
                continue

            q = get_qualifying_genes(methods_dict, cluster_name,
                                     pattern_name, thresholds)

            fig, axes = plt.subplots(1, len(active),
                                     figsize=(5 * len(active), 4.5),
                                     squeeze=False)

            for i, mname in enumerate(active):
                ax = axes[0, i]
                genes = q[mname].index.tolist()
                n_genes = len(genes)
                for gene in genes[:max_plot]:
                    if gene in gene_df.index:
                        ax.plot(x, gene_df.loc[gene], color='steelblue',
                                alpha=0.3, linewidth=0.8)
                ax.plot(x, pattern_vals, color='crimson', linewidth=2.5,
                        linestyle='--')
                op, val = thresholds[mname]
                fq_val = fit_quality[cluster_name][mname]
                ax.set_title(f'{mname} ({op}{val})\n'
                             f'{n_genes:,} genes | fit MSE={fq_val:.4f}',
                             fontsize=10)
                ax.set_xlabel('Timepoint')
                ax.set_xticks(x)
                if i == 0:
                    ax.set_ylabel('Gene counts')
                ax.grid(alpha=0.2)

            all_silenced = sil1 + sil2
            sil_str = ', '.join(all_silenced) if all_silenced else 'none'
            fig.suptitle(f'{cluster_name} (LT) — Option D\n'
                         f'Silenced: {sil_str}',
                         fontsize=12, fontweight='bold')
            plt.tight_layout()
            plt.show()


plot_qualifying_genes_optionD(
    all_methodsLT, constraint_patternsLT, THRESHOLDS,
    fit_quality, FIT_THRESHOLD, lt=True
)

## 6. Gated consensus using only double-gated active methods

Identical to the base gated consensus, except the method pool is
restricted to methods that pass **both** gates.

In [ ]:
def gated_consensus_optionD(methods_dict, pat_dict, thresholds,
                            fit_quality, fit_threshold,
                            cluster_name, pattern_name='constraint',
                            top_n=20):
    """Score-gated + fit-gated ensemble.

    Returns (result_df, active, silenced_stage1, silenced_stage2).
    """
    active, sil1, sil2 = get_active_methods(
        methods_dict, pat_dict, thresholds,
        fit_quality, fit_threshold, cluster_name
    )

    if len(active) == 0:
        return pd.DataFrame(), active, sil1, sil2

    q = get_qualifying_genes(methods_dict, cluster_name, pattern_name, thresholds)

    # Build rank df for active methods only
    rank_df = pd.DataFrame()
    for mname in active:
        scores, ascending = methods_dict[mname]
        s = scores[cluster_name][pattern_name]
        rank_df[mname] = s.rank(ascending=ascending, method='min')

    # A gene gets a vote from a method only if it's in that method's qualifying set
    vote_df = pd.DataFrame(index=rank_df.index)
    for mname in active:
        qualifying_genes = set(q[mname].index)
        vote_df[mname] = rank_df.index.isin(qualifying_genes).astype(int)

    votes = vote_df.sum(axis=1)
    mean_rank = rank_df[active].mean(axis=1)

    result = pd.DataFrame({
        'votes': votes,
        'max_possible': len(active),
        'mean_rank': mean_rank,
    })
    result = result[result['votes'] > 0]
    result = result.sort_values(['votes', 'mean_rank'], ascending=[False, True])

    return result.head(top_n), active, sil1, sil2


# ── Results table ─────────────────────────────────────────────────────
print('OPTION D — DOUBLE-GATED CONSENSUS ENSEMBLE (LT DATA)')
print('=' * 90)

for cluster_name in cluster_names:
    result, active, sil1, sil2 = gated_consensus_optionD(
        all_methodsLT, constraint_patternsLT, THRESHOLDS,
        fit_quality, FIT_THRESHOLD,
        cluster_name, top_n=TOP_N
    )
    print(f'\n--- {cluster_name} ---')
    print(f'  Active:           {len(active)}/{len(method_names)} — '
          f'{", ".join(active) if active else "NONE"}')
    if sil1:
        print(f'  Silenced stage 1: {", ".join(sil1)}')
    if sil2:
        print(f'  Silenced stage 2: {", ".join(sil2)}')

    if result.empty:
        print('  NO QUALIFYING GENES — loosen thresholds or FIT_THRESHOLD')
        continue

    q = get_qualifying_genes(all_methodsLT, cluster_name, 'constraint', THRESHOLDS)

    print(f'\n  Rank  Gene                 Votes/{len(active)}  Mean Rank  Qualified in')
    print(f'  {"-" * 78}')
    for i, (gene, row) in enumerate(result.iterrows(), 1):
        v = int(row['votes'])
        mr = row['mean_rank']
        in_methods = [m for m in active if gene in q[m].index]
        print(f'  {i:<6}{gene:<20} {v:>5}/{len(active)}  {mr:>9.1f}  '
              f'{", ".join(in_methods)}')

## 7. Ensemble plots

Top-N genes from the double-gated consensus, with opacity proportional
to vote count.

In [ ]:
def plot_gated_consensus_optionD(methods_dict, pat_dict, thresholds,
                                 fit_quality, fit_threshold,
                                 lt=True, top_n=20):
    """Plot the double-gated consensus top-N genes per cluster."""
    x = [0, 3, 6, 9]
    suffix = 'LT' if lt else ''

    for cluster_name, pattern_dict in pat_dict.items():
        gene_file = f'analysis data/gene_counts{suffix}/{cluster_name}_annotated.csv'
        gene_df = pd.read_csv(gene_file, index_col=0)
        gene_df.columns = x

        for pattern_name, pattern_vals in pattern_dict.items():
            result, active, sil1, sil2 = gated_consensus_optionD(
                methods_dict, pat_dict, thresholds,
                fit_quality, fit_threshold,
                cluster_name, pattern_name, top_n=top_n
            )
            if result.empty:
                print(f'{cluster_name}: no ensemble genes (all methods silenced)')
                continue

            max_votes = len(active)
            fig, ax = plt.subplots(figsize=(10, 6))

            for gene in result.index:
                if gene in gene_df.index:
                    v = result.loc[gene, 'votes']
                    alpha = 0.2 + 0.6 * (v / max_votes)
                    ax.plot(x, gene_df.loc[gene], color='steelblue',
                            alpha=alpha, linewidth=1)

            ax.plot(x, pattern_vals, color='crimson', linewidth=2.5,
                    linestyle='--')

            all_silenced = sil1 + sil2
            sil_str = ', '.join(all_silenced) if all_silenced else 'none'
            title = (f'Option D Consensus | {cluster_name} (LT) | '
                     f'Top {top_n} genes\n'
                     f'Active: {", ".join(active)} | Silenced: {sil_str}')
            ax.set_title(title, fontsize=10)
            ax.set_xlabel('Timepoint')
            ax.set_ylabel('Gene counts')
            ax.set_xticks(x)

            legend_elements = [
                Line2D([0], [0], color='crimson', linewidth=2.5,
                       linestyle='--', label='Constraint pattern'),
                Line2D([0], [0], color='steelblue', alpha=0.8,
                       label=f'High consensus (>={max_votes - 1}/{max_votes} votes)'),
                Line2D([0], [0], color='steelblue', alpha=0.3,
                       label='Low consensus (1 vote)'),
            ]
            ax.legend(handles=legend_elements, loc='upper right')
            ax.grid(alpha=0.2)
            plt.tight_layout()
            plt.show()


plot_gated_consensus_optionD(
    all_methodsLT, constraint_patternsLT, THRESHOLDS,
    fit_quality, FIT_THRESHOLD, lt=True, top_n=TOP_N
)